# <center> False-Bottom Encryption (FBE) </center>

In [5]:
"""
False-Bottom Encryption: Deniable Encryption from Secret Sharing
================================================================
Generic implementation supporting any number of messages.

Based on: Ahmad, Rass, Schartner - IEEE Access, 2023
DOI: 10.1109/ACCESS.2023.3288285

This implementation fixes several bugs from the original notebook:
  - Proper index tracking (no c.index() collisions on duplicate values)
  - Always appends new alpha to ciphertext (no silent skipping of duplicates)
  - Uses pow(a, -1, p) for modular inverse instead of brute-force loop
  - Supports arbitrary number of messages via a single class interface
"""

import random
from dataclasses import dataclass, field
from typing import List, Tuple, Optional


@dataclass
class SecretKey:
    """
    A decryption key sk_i for message m_i.

    Stores pairs of (ciphertext_index, keybase_index) that define
    which elements of c and r are multiplied and summed to recover m_i.
    """
    pairs: List[Tuple[int, int]]  # list of (j, rho) pairs

    def __repr__(self):
        return f"SK({self.pairs})"


class FalseBottomEncryption:
    """
    False-Bottom Encryption scheme from secret sharing.

    Supports:
      - Init:   create empty ciphertext and key base
      - Enc:    add any number of messages over time
      - Dec:    decrypt any message given its secret key
      - Update: change an existing message inside the ciphertext
      - Del:    delete a message by overwriting with random data

    All arithmetic is done in Z_p for a prime p.
    """

    def __init__(self, p: int = 9973, k: int = 5, seed: Optional[int] = None):
        """
        System Initialization: (c, r) <- Init(t, n, k)

        Args:
            p:    prime defining the field Z_p (determines block size t ~ log2(p))
            k:    size of the key base and initial ciphertext
            seed: optional RNG seed for reproducibility
        """
        if seed is not None:
            random.seed(seed)

        self.p = p
        self.k = k

        # Key base (root key): r = (r_1, ..., r_k) in F* (nonzero)
        self.r = [random.randint(1, p - 1) for _ in range(k)]

        # Empty ciphertext: c = (alpha_1, ..., alpha_k) in F^k
        self.c = [random.randint(1, p - 1) for _ in range(k)]

        # Track all secret keys for update/delete operations
        self._secret_keys: List[SecretKey] = []
        self._messages: List[int] = []  # stored for verification only

    def _mod_inv(self, a: int) -> int:
        """Modular inverse of a mod p using Python's built-in pow."""
        return pow(a, -1, self.p)

    def encrypt(self, message: int) -> SecretKey:
        """
        Enc(c, m): Add a message to the existing ciphertext.

        Returns the secret key needed to later decrypt this message.

        The ciphertext grows by exactly 1 element per message added.

        Args:
            message: integer in [0, p-1] to encrypt

        Returns:
            SecretKey for decrypting this message
        """
        if not (0 <= message < self.p):
            raise ValueError(f"Message must be in [0, {self.p - 1}], got {message}")

        p = self.p
        k = self.k
        c = self.c

        # Pick how many terms to use: n_i in {2, ..., k}
        n_i = random.randint(2, k)

        # Pick n_i key-base indices (which r-values to use)
        rho_indices = sorted(random.sample(range(k), n_i))

        # Pick (n_i - 1) ciphertext indices from existing c.
        # IMPORTANT: avoid the "unique" (last-appended) index of any existing
        # message, so that updates to one message don't break others.
        # This enforces condition (2) from the paper (the "forbidden set F").
        forbidden = {sk.pairs[-1][0] for sk in self._secret_keys}
        available = [idx for idx in range(len(c)) if idx not in forbidden]

        if len(available) < n_i - 1:
            # Not enough non-forbidden elements; reduce n_i
            n_i = len(available) + 1
            if n_i < 2:
                raise RuntimeError(
                    "Not enough available ciphertext elements. "
                    "Increase k or the initial ciphertext size."
                )
            # Re-sample key-base indices with the adjusted n_i
            rho_indices = sorted(random.sample(range(k), n_i))
            r_values = [self.r[rho] for rho in rho_indices]

        j_indices = sorted(random.sample(available, n_i - 1))

        # Get the actual r-values and alpha-values
        r_values = [self.r[rho] for rho in rho_indices]
        alpha_values = [c[j] for j in j_indices]

        # Compute the new alpha that makes the equation hold:
        # m = alpha_1 * r_1 + ... + alpha_{n-1} * r_{n-1} + alpha_new * r_n
        # => alpha_new = r_n^{-1} * (m - sum(alpha_j * r_j for j=1..n-1)) mod p
        partial_sum = sum(a * r for a, r in zip(alpha_values, r_values[:-1])) % p
        alpha_new = (self._mod_inv(r_values[-1]) * (message - partial_sum)) % p

        # Append the new alpha to the ciphertext
        new_index = len(c)
        c.append(alpha_new)

        # Build the secret key: pairs of (ciphertext_index, keybase_index)
        pairs = list(zip(j_indices, rho_indices[:-1]))
        pairs.append((new_index, rho_indices[-1]))

        sk = SecretKey(pairs=pairs)
        self._secret_keys.append(sk)
        self._messages.append(message)

        return sk

    def decrypt(self, sk: SecretKey) -> int:
        """
        Dec(c, sk): Decrypt a message using its secret key.

        Computes m = sum(c[j] * r[rho] for (j, rho) in sk) mod p

        Args:
            sk: the secret key returned by encrypt()

        Returns:
            the decrypted message
        """
        p = self.p
        result = 0
        for (j, rho) in sk.pairs:
            result = (result + self.c[j] * self.r[rho]) % p
        return result

    def update(self, message_index: int, new_message: int) -> None:
        """
        Update(c, i, sk_1, ..., sk_l, m'): Replace message m_i with m'.

        This modifies the ciphertext element that is unique to message i
        (the last element appended when m_i was encrypted), recomputing
        it so the equation yields new_message instead.

        All other messages remain correctly decryptable because the unique
        element is used only by message i (condition (2) in the paper).

        Args:
            message_index: which message to update (0-based)
            new_message:   the replacement message
        """
        if not (0 <= new_message < self.p):
            raise ValueError(f"Message must be in [0, {self.p - 1}]")
        if not (0 <= message_index < len(self._secret_keys)):
            raise ValueError(f"Message index {message_index} out of range")

        p = self.p
        sk = self._secret_keys[message_index]

        # The last pair in sk points to the unique alpha for this message
        unique_j, unique_rho = sk.pairs[-1]

        # Recompute: new_alpha = r[unique_rho]^{-1} * (m' - sum of other terms)
        partial_sum = 0
        for (j, rho) in sk.pairs[:-1]:
            partial_sum = (partial_sum + self.c[j] * self.r[rho]) % p

        new_alpha = (self._mod_inv(self.r[unique_rho]) * (new_message - partial_sum)) % p
        self.c[unique_j] = new_alpha
        self._messages[message_index] = new_message

    def delete(self, message_index: int) -> None:
        """
        Del(c, i, sk_1, ..., sk_l): Delete message m_i from c.

        Overwrites m_i with random data (indistinguishable from an update)
        and discards the secret key.

        Args:
            message_index: which message to delete (0-based)
        """
        # Overwrite with random message
        random_msg = random.randint(0, self.p - 1)
        self.update(message_index, random_msg)
        # Mark as deleted (key is now useless since it decrypts to garbage)
        self._messages[message_index] = None

    def get_state_summary(self) -> str:
        """Return a human-readable summary of the current state."""
        lines = []
        lines.append(f"Prime p = {self.p}")
        lines.append(f"Key base size k = {self.k}")
        lines.append(f"Key base r = {self.r}")
        lines.append(f"Ciphertext c ({len(self.c)} elements) = {self.c}")
        lines.append(f"Messages stored: {len(self._secret_keys)}")
        for i, (sk, m) in enumerate(zip(self._secret_keys, self._messages)):
            status = "DELETED" if m is None else f"m = {m}"
            lines.append(f"  Message {i}: {status}, key = {sk}")
        return "\n".join(lines)


# =============================================================================
# Helper: safe integer input with validation
# =============================================================================

def _input_int(prompt: str, min_val: int = None, max_val: int = None) -> int:
    """Prompt for an integer, repeating until a valid value is given."""
    while True:
        try:
            val = int(input(prompt))
            if min_val is not None and val < min_val:
                print(f"  Value must be at least {min_val}. Try again.")
                continue
            if max_val is not None and val > max_val:
                print(f"  Value must be at most {max_val}. Try again.")
                continue
            return val
        except ValueError:
            print("  Invalid input. Please enter an integer.")


def _print_separator():
    print("-" * 60)


def _print_header(title: str):
    print()
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)


# =============================================================================
# Interactive mode
# =============================================================================

def interactive():
    _print_header("FALSE-BOTTOM ENCRYPTION")
    print("  Deniable Encryption from Secret Sharing")
    print("  Supports any number of messages")
    print("=" * 60)

    # -------------------------------------------------------------------------
    # STEP 1: System Initialization
    # -------------------------------------------------------------------------
    _print_header("STEP 1: SYSTEM INITIALIZATION")

    p = _input_int("Enter the prime p (e.g. 9973): ", min_val=3)
    k = _input_int("Enter the key base size k (e.g. 5): ", min_val=2)

    fbe = FalseBottomEncryption(p=p, k=k)

    print(f"\n  Prime p = {p}")
    print(f"  Key base size k = {k}")
    print(f"  Key base r = {fbe.r}")
    print(f"  Initial ciphertext c = {fbe.c}")

    # Track messages and keys for user reference
    messages = []       # original plaintext values
    secret_keys = []    # corresponding SecretKey objects

    # -------------------------------------------------------------------------
    # STEP 2: Encryption
    # -------------------------------------------------------------------------
    _print_header("STEP 2: ENCRYPTION")

    num_encrypt = _input_int(
        "How many messages do you want to encrypt? ", min_val=1
    )

    print(f"\n  Enter {num_encrypt} message(s), each an integer in [0, {p - 1}]:\n")

    for i in range(num_encrypt):
        msg = _input_int(f"  m_{i} = ", min_val=0, max_val=p - 1)
        sk = fbe.encrypt(msg)
        messages.append(msg)
        secret_keys.append(sk)
        print(f"    Encrypted. Key: {sk}")

    _print_separator()
    print(f"\n  Ciphertext ({len(fbe.c)} elements): {fbe.c}")

    # -------------------------------------------------------------------------
    # STEP 3: Decryption (verify all)
    # -------------------------------------------------------------------------
    _print_header("STEP 3: DECRYPTION (verifying all messages)")

    all_ok = True
    for i in range(len(messages)):
        if messages[i] is None:
            print(f"  m_{i}: DELETED")
            continue
        recovered = fbe.decrypt(secret_keys[i])
        match = recovered == messages[i]
        status = "OK" if match else "FAIL"
        if not match:
            all_ok = False
        print(f"  m_{i}: expected = {messages[i]}, decrypted = {recovered}  [{status}]")

    if all_ok:
        print("\n  All decryptions correct!")
    else:
        print("\n  WARNING: Some decryptions failed!")

    # -------------------------------------------------------------------------
    # STEP 4: Update messages
    # -------------------------------------------------------------------------
    _print_header("STEP 4: UPDATE MESSAGES")

    # Show current message list for reference
    print("  Current messages:")
    for i in range(len(messages)):
        tag = "DELETED" if messages[i] is None else str(messages[i])
        print(f"    m_{i} = {tag}")
    print()

    num_update = _input_int(
        "How many messages do you want to update? (0 to skip): ", min_val=0
    )

    for u in range(num_update):
        print(f"\n  Update {u + 1} of {num_update}:")
        idx = _input_int(
            f"    Which message index to update? [0-{len(messages) - 1}]: ",
            min_val=0, max_val=len(messages) - 1
        )
        if messages[idx] is None:
            print("    That message was deleted. Choose another.")
            idx = _input_int(
                f"    Which message index to update? [0-{len(messages) - 1}]: ",
                min_val=0, max_val=len(messages) - 1
            )

        new_msg = _input_int(
            f"    New value for m_{idx} [0-{p - 1}]: ",
            min_val=0, max_val=p - 1
        )

        old_msg = messages[idx]
        fbe.update(idx, new_msg)
        messages[idx] = new_msg

        recovered = fbe.decrypt(secret_keys[idx])
        print(f"    Updated m_{idx}: {old_msg} -> {new_msg}")
        print(f"    Verification: decrypt(sk_{idx}) = {recovered}  "
              f"[{'OK' if recovered == new_msg else 'FAIL'}]")

    # Verify all other messages are still intact after updates
    if num_update > 0:
        _print_separator()
        print("\n  Verifying all remaining messages after updates:")
        for i in range(len(messages)):
            if messages[i] is None:
                print(f"    m_{i}: DELETED")
                continue
            recovered = fbe.decrypt(secret_keys[i])
            status = "OK" if recovered == messages[i] else "FAIL"
            print(f"    m_{i} = {messages[i]}, decrypted = {recovered}  [{status}]")

    # -------------------------------------------------------------------------
    # STEP 5: Delete messages
    # -------------------------------------------------------------------------
    _print_header("STEP 5: DELETE MESSAGES")

    # Show current state
    active = [(i, m) for i, m in enumerate(messages) if m is not None]
    print(f"  Active messages: {len(active)}")
    for i, m in active:
        print(f"    m_{i} = {m}")
    print()

    num_delete = _input_int(
        "How many messages do you want to delete? (0 to skip): ",
        min_val=0, max_val=len(active)
    )

    for d in range(num_delete):
        print(f"\n  Delete {d + 1} of {num_delete}:")

        # Show only active messages
        active = [(i, m) for i, m in enumerate(messages) if m is not None]
        if not active:
            print("    No active messages left to delete.")
            break

        print("    Active messages:")
        for i, m in active:
            print(f"      m_{i} = {m}")

        idx = _input_int(
            f"    Which message index to delete? ",
            min_val=0, max_val=len(messages) - 1
        )
        if messages[idx] is None:
            print(f"    m_{idx} is already deleted. Skipping.")
            continue

        old_val = messages[idx]
        fbe.delete(idx)
        garbage = fbe.decrypt(secret_keys[idx])
        messages[idx] = None

        print(f"    Deleted m_{idx} (was {old_val}).")
        print(f"    Decryption with old key now returns: {garbage} (garbage)")

    # -------------------------------------------------------------------------
    # STEP 6: Final state
    # -------------------------------------------------------------------------
    _print_header("FINAL STATE")

    print(f"  Ciphertext ({len(fbe.c)} elements): {fbe.c}\n")

    print("  Message summary:")
    for i in range(len(messages)):
        if messages[i] is None:
            print(f"    m_{i}: DELETED")
        else:
            recovered = fbe.decrypt(secret_keys[i])
            status = "OK" if recovered == messages[i] else "FAIL"
            print(f"    m_{i} = {messages[i]}, key = {secret_keys[i]}, "
                  f"decrypt = {recovered} [{status}]")

    print(f"\n  Total ciphertext elements: {len(fbe.c)}")
    active_count = sum(1 for m in messages if m is not None)
    print(f"  Active messages: {active_count}")
    print(f"  Deleted messages: {len(messages) - active_count}")

    print("\n" + "=" * 60)
    print("  DONE")
    print("=" * 60)


# =============================================================================
# Automated test (non-interactive)
# =============================================================================

def automated_test():
    """Run a quick automated correctness check without any user input."""
    print("Running automated tests...\n")

    # Test 1: basic encrypt/decrypt with many messages
    fbe = FalseBottomEncryption(p=9973, k=5, seed=42)
    msgs = [100, 200, 300, 42, 7777, 1234, 5555, 9000, 13, 8080]
    keys = [fbe.encrypt(m) for m in msgs]

    failures = sum(1 for m, sk in zip(msgs, keys) if fbe.decrypt(sk) != m)
    print(f"  [Test 1] Encrypt/decrypt {len(msgs)} messages: "
          f"{len(msgs) - failures}/{len(msgs)} correct")

    # Test 2: update should not break other messages
    fbe.update(2, 999)
    assert fbe.decrypt(keys[2]) == 999, "Update failed"
    for i, (m, sk) in enumerate(zip(msgs, keys)):
        if i == 2:
            continue
        assert fbe.decrypt(sk) == m, f"Update broke m_{i}"
    print("  [Test 2] Update m_2 -> 999: OK, others unaffected")

    # Test 3: delete
    fbe.delete(4)
    assert fbe.decrypt(keys[4]) != msgs[4], "Delete did not change decryption"
    print(f"  [Test 3] Delete m_4: OK (decrypts to garbage)")

    # Test 4: scale to 1000 messages
    fbe2 = FalseBottomEncryption(p=104729, k=8, seed=999)
    scale_msgs = [random.randint(0, 104728) for _ in range(1000)]
    scale_keys = [fbe2.encrypt(m) for m in scale_msgs]
    failures = sum(1 for m, sk in zip(scale_msgs, scale_keys) if fbe2.decrypt(sk) != m)
    print(f"  [Test 4] Scale test (1000 messages, p=104729): "
          f"{1000 - failures}/1000 correct")

    print("\nAll tests passed!")


# =============================================================================
# Entry point
# =============================================================================

if __name__ == "__main__":
    print("\nFalse-Bottom Encryption")
    print("  1. Interactive mode (user prompts)")
    print("  2. Automated test (no input needed)")
    choice = input("\nSelect mode [1/2]: ").strip()

    if choice == "2":
        automated_test()
    else:
        interactive()


False-Bottom Encryption
  1. Interactive mode (user prompts)
  2. Automated test (no input needed)



Select mode [1/2]:  1



  FALSE-BOTTOM ENCRYPTION
  Deniable Encryption from Secret Sharing
  Supports any number of messages

  STEP 1: SYSTEM INITIALIZATION


Enter the prime p (e.g. 9973):  9973
Enter the key base size k (e.g. 5):  10



  Prime p = 9973
  Key base size k = 10
  Key base r = [8721, 1640, 7314, 5487, 4668, 5482, 4903, 1073, 5078, 4189]
  Initial ciphertext c = [5591, 8759, 1449, 7025, 1349, 6651, 6322, 8727, 5411, 6573]

  STEP 2: ENCRYPTION


...


  Delete 1 of 2:
    Active messages:
      m_0 = 200
      m_1 = 101
      m_2 = 102
      m_3 = 103
      m_4 = 204
      m_5 = 105
      m_6 = 106
      m_7 = 207
      m_8 = 108
      m_9 = 109
      m_10 = 110
      m_11 = 111
      m_12 = 112
      m_13 = 113
      m_14 = 114


    Which message index to delete?  6


    Deleted m_6 (was 106).
    Decryption with old key now returns: 592 (garbage)

  Delete 2 of 2:
    Active messages:
      m_0 = 200
      m_1 = 101
      m_2 = 102
      m_3 = 103
      m_4 = 204
      m_5 = 105
      m_7 = 207
      m_8 = 108
      m_9 = 109
      m_10 = 110
      m_11 = 111
      m_12 = 112
      m_13 = 113
      m_14 = 114


    Which message index to delete?  13


    Deleted m_13 (was 113).
    Decryption with old key now returns: 1243 (garbage)

  FINAL STATE
  Ciphertext (25 elements): [5591, 8759, 1449, 7025, 1349, 6651, 6322, 8727, 5411, 6573, 6765, 5356, 2123, 1664, 9970, 2340, 1314, 6583, 1970, 2779, 5405, 6100, 175, 7903, 6207]

  Message summary:
    m_0 = 200, key = SK([(6, 0), (8, 5), (9, 8), (10, 9)]), decrypt = 200 [OK]
    m_1 = 101, key = SK([(2, 0), (6, 1), (9, 2), (11, 3)]), decrypt = 101 [OK]
    m_2 = 102, key = SK([(0, 0), (1, 1), (3, 2), (4, 3), (5, 4), (6, 5), (7, 6), (8, 7), (9, 8), (12, 9)]), decrypt = 102 [OK]
    m_3 = 103, key = SK([(1, 0), (2, 1), (3, 2), (4, 3), (6, 4), (7, 5), (8, 6), (9, 8), (13, 9)]), decrypt = 103 [OK]
    m_4 = 204, key = SK([(2, 1), (4, 3), (6, 6), (14, 8)]), decrypt = 204 [OK]
    m_5 = 105, key = SK([(5, 0), (7, 2), (15, 7)]), decrypt = 105 [OK]
    m_6: DELETED
    m_7 = 207, key = SK([(3, 0), (5, 4), (7, 7), (17, 8)]), decrypt = 207 [OK]
    m_8 = 108, key = SK([(1, 0), (2, 2), (4, 3), (6, 